# SLR09 — Fine-tune NARA, 15 label + adapter

**Workflow:** periksa sumber/split → baseline v1 aktual → fine-tune 157 train → pilih checkpoint dari 41 validation → evaluasi → ekspor model dan ZIP hasil.

**Run all pada notebook ini benar-benar menjalankan fine-tune.** Pilih **Runtime → Change runtime type → GPU**, misalnya T4. Data dan kode sudah tertanam; tidak perlu mengunggah Excel lagi. Tidak ada flag neural yang harus diubah menjadi True.

Sumber bobot tetap `SILAR_NARA_v1.zip`. Dataset pilot: 198 narasi, 157 train/41 validation, semua 15 label. `KEB-00486` tetap TUNDA. Keputusan A dipakai: WOUND juga mencakup kondisi fisik eksplisit seperti kritis, sesak napas, dan tidak sadarkan diri. Label manusia tidak diubah.

Ini eksperimen fine-tune dari model yang sudah ada, bukan klaim membuat model dari nol. Notebook sumber training NARA lama tersedia pada paket kode. Belum ada metrik fine-tune pada file notebook yang belum dijalankan.


## 1. Library
Gunakan sesi GPU baru. Jika library sudah pernah diimpor dengan versi lain, restart sesi setelah instalasi.

In [ ]:
# Library tambahan saja; torch bawaan runtime GPU Colab dipakai.
# Versi Trainer dipatok agar argumen dan perilaku API dapat ditelusuri.
import os, subprocess, sys
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.1", "accelerate==1.10.1", "tokenizers==0.22.1", "safetensors>=0.4.3",
    "scikit-learn", "matplotlib"])
print("Library siap. Torch/CUDA bawaan tidak diganti oleh sel ini.")


## 2. Drive dan konfigurasi
Dua ZIP sumber yang sudah ada di Drive dipakai ulang. SENA tidak dilatih pada notebook ini.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# Folder aset SLR09 yang telah dipakai sebelumnya.
DRIVE_FOLDER = Path("/content/drive/MyDrive/SILAR_SLR09")
WORK_ROOT = Path("/content/silar_nara_finetune_v01")

# Mengikuti konfigurasi historis NARA. Logging dibuat per epoch agar kurva jelas.
CONFIG = {"seed": 42, "max_length": 256, "learning_rate": 2e-5,
          "train_batch_size": 16, "eval_batch_size": 32,
          "max_epochs": 10, "weight_decay": 0.01,
          "early_stopping_patience": 2,
          "metric_for_best_model": "macro_f1_no_o",
          "logging_strategy": "epoch", "transformers_version": "4.57.1"}

for name in ("SILAR_NARA_v1.zip", "Semantic Role Labeling Datasets for Crisis Event.zip"):
    if not (DRIVE_FOLDER / name).is_file():
        raise FileNotFoundError(f"Belum ditemukan: {DRIVE_FOLDER / name}. Sesuaikan DRIVE_FOLDER.")
WORK_ROOT.mkdir(parents=True, exist_ok=True)


## 3. Kode dan data mandiri
Semua file diverifikasi dengan SHA-256. Bobot dibaca lokal; tidak diunduh ulang dari model hub.

In [ ]:
# Kode/data yang sama tersedia sebagai file lengkap pada ZIP pendamping.
import base64, hashlib, json, sys, zlib
EMBEDDED = json.loads(zlib.decompress(base64.b64decode('...')))
EXPECTED_SHA256 = {'code/nara_experiment.py': '734e9c15a13ac967a72a482c68a9fc777e6632a75957ffd854b0863894a28409', 'code/nara_data.py': 'eea1124afd1d0628b7a96c3d9dbca43a116f8bed0fef76f2c67718e7ff2e4a29', 'code/nara_metrics.py': '8f74d058db3e613c77df83c772b5142f2b504dfde542455284754e4c8ceefcb7', 'code/slr09_runtime.py': '95d48353cd4fec980f8d4795d744cb0543827c94f22b960b0e5b38470409f66d', 'code/silar_ai_adapter.py': '1dead87540803429634cad961c4ce6ee192447d3d4610630f7105e221939b99c', 'code/silar_nara_composition.py': '10a560b2bd8e14fb069695525023a43828638d4dd21ad721886c1fce1c852885', 'code/silar_postprocessing.py': '38aaf3d18274a79bf056aac3591e922ae015f2d403a3eb0ed9a71060c1197abd', 'code/silar_victim_guard.py': '7cdc7b355130252073e734f97ee65e0692d089cafdadc7eb0036c5806927e4e7', 'code/legacy_postprocessing.py': '06275b3aa338718689fba66984626754e6b34739156f65e378ceaad4b06bfa9f', 'code/asset_lock.json': '31fbe639ea58f7c235ff0a8243704a09ad92f032390444aa961db6d4569b2db7', 'data/asset_lock.json': '31fbe639ea58f7c235ff0a8243704a09ad92f032390444aa961db6d4569b2db7', 'data/cached_neural_predictions.jsonl': '5a5888b6eb249981e59b5ab8c04da87f287ce2f470532e2ec202cc84205426a5', 'data/dataset_manifest.json': '3d93d3a40aaf540db6af2301e437401763c7973d832e585eb258b49004cc6ab6', 'data/diagnostic_cases.jsonl': '1e3c1ca0c407121b454ccbae2b76436c1641dbe1f4cf2279cd203a1065133633', 'data/excluded_supplement_review.jsonl': '6f3e4521b36c01f6f8360f79e5ee68358bea805b4be0018a8e4ec7b360437bea', 'data/nara_reviewed_84_fields.json': '41440b47e9aabdab8d9dc7205a3f48977f706edd5b5b25313a098afb9f44bf89', 'data/train_reviewed.jsonl': '9ebbf927e78948c2387f34d21f58f728f3d99db747798083f77efb69d0524548', 'data/validation_reviewed.jsonl': '32edb43f7b9cc681870cbfad1417ef9aa7f409aacd890f39f86f9e775604a509', 'splits/test_ids.json': 'de6315984bcf210a945e8d9e55490c0a5c9802d5485e768fe621b2ab2df0660d', 'splits/train_ids.json': 'ec59d172965775aa417b6f6060baa9e75efcb2284dc1f6284e709003f5202860', 'splits/validation_ids.json': '1b7550e0ee24ac57b26a375477f23c6210d8d051b8caa33773cd6309c3743e40', 'SLR09_Pedoman_NARA_15Label_v0.2.md': '379954d86d7097b0ccea069a2946d4dfb7633f55cb19e63f5cef4c9ac2079451'}
for relative, value in EMBEDDED.items():
    raw = value.encode("utf-8")
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256[relative]:
        raise ValueError("Isi notebook berbeda: " + relative)
    target = WORK_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(raw)
sys.path.insert(0, str(WORK_ROOT / "code"))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
from nara_experiment import NaraExperiment
experiment = NaraExperiment(WORK_ROOT, DRIVE_FOLDER, WORK_ROOT, config=CONFIG)
print("Run baru:", experiment.run_id)


## 4. Verifikasi data, aset, GPU, dan alignment
Special token dan piece lanjutan memakai -100. Perbedaan token korpus vs regex serving dicatat. Tidak mengubah pembagian train/validation.

In [ ]:
prepared = experiment.prepare()
print({key: prepared[key] for key in ('train_narratives', 'validation_narratives', 'historical_test_narratives')})


## 5. Baseline NARA v1 aktual
Evaluasi pada 41 validation yang sama. Diagnostik tujuh field memakai inferensi NARA baru dan prediksi SENA aktual tersimpan sebagai komponen tetap.

In [ ]:
baseline_metrics = experiment.baseline()
print(baseline_metrics)


## 6. Fine-tune
Maksimal 10 epoch, early stopping patience 2, checkpoint terbaik berdasarkan macro F1 seluruh 14 label non-O. Test dan 12 diagnostik tidak menentukan checkpoint.

In [ ]:
training_state = experiment.fit()
print({key: training_state[key] for key in ('training_executed', 'training_completed', 'optimizer_steps', 'best_validation_macro_f1_no_o')})


## 7. Evaluasi hasil
Token accuracy/F1, classification report, confusion matrix 15 label, dan exact span flat. Test 1.000 memakai label historis yang belum diselaraskan ke pedoman v0.2; laporkan terpisah. Empat contoh kondisi fisik ditandai train/validation, bukan semuanya heldout.

In [ ]:
comparison = experiment.evaluate()
print(json.dumps(comparison, ensure_ascii=False, indent=2))


## 8. Ekspor dan simpan hasil
Model diuji reload. ZIP model tersimpan di Drive; ZIP hasil yang lebih kecil berisi metrik, prediksi, grafik, dan audit. Model produksi tidak otomatis diganti.

In [ ]:
saved = experiment.export()
print(json.dumps(saved, ensure_ascii=False, indent=2))


## Yang dikirim kembali

Kirim **`SLR09_NARA_Hasil_FineTune_<run_id>.zip`** dari folder Drive. Bobot model tidak perlu diunggah ke percakapan. Jika terjadi error, ZIP hasil sementara dan `run_state.json` mencatat tahap serta apakah training sudah sempat berjalan; kirim ZIP tersebut untuk diperiksa.

Nilai validation adalah hasil development pilot yang dipakai memilih checkpoint. Diagnostik D01–D12 sudah dikenal dan ikut menghitung field kosong; hasilnya bukan akurasi produksi. Prediksi SENA pada diagnostik berasal dari run lama yang benar-benar dijalankan, bukan inferensi SENA baru. Gold test historis mempunyai definisi berbeda dari target revisi.

Referensi API: [Trainer v4.57.1](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer), [token classification](https://github.com/huggingface/transformers/blob/v4.57.1/docs/source/en/tasks/token_classification.md), dan [callbacks](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/callback).
